# 02 — Baseline Multimodal (hasta 10 imágenes)

Fine-tuning de **Qwen2.5-VL-3B-Instruct** con LoRA para detección de datos de contacto en listings MeLi.

| Parámetro | Valor |
|-----------|-------|
| Modelo base | Qwen2.5-VL-3B-Instruct (4-bit) |
| LoRA rank / alpha | r=8 / α=16 |
| GPU objetivo | T4 (14.5 GB) |
| Imágenes / listing | hasta 10 |
| Métrica principal | F2-score |

**Prerequisito:** haber corrido `01_build_splits.ipynb` para tener `train.csv` y `val.csv` en Drive.

**Estrategia de imágenes:**
- Drive (`DRIVE_IMGS_DIR`): descarga persistente, una sola vez
- Disco local (`LOCAL_IMGS_DIR`): copia rápida al inicio de cada sesión, training desde acá

## 0 — Setup y dependencias

In [ ]:
# Setup: monta Drive, clona/actualiza repo, instala deps base
exec(open('/content/drive/MyDrive/contact-detection/scripts/colab_setup.py').read())

In [ ]:
# Instalar Unsloth (requiere restart del kernel la primera vez)
import subprocess
subprocess.run(['pip', 'install', '-q', 'unsloth'], check=True)
subprocess.run(['pip', 'install', '--upgrade', '-q', 'unsloth_zoo'], check=True)
print('✅ Unsloth instalado')

In [ ]:
# ── Patch para KeyError: 'sanitize_logprob' (bug unsloth/unsloth_zoo en Colab) ──
try:
    from unsloth_zoo import rl_replacements as _rl
    if 'sanitize_logprob' not in _rl.RL_REPLACEMENTS:
        _rl.RL_REPLACEMENTS['sanitize_logprob'] = lambda logprobs, *a, **kw: logprobs
        print('⚠️  Patch aplicado: sanitize_logprob')
except Exception:
    pass

# ── Unsloth PRIMERO (antes que torch y transformers) ──
from unsloth import FastVisionModel

import os, gc, json, sys
import torch
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import yaml
from pathlib import Path

from transformers import AutoProcessor, Trainer, TrainingArguments

REPO_DIR = '/content/meli-contact-detection'
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

from src.data.dataset      import csv_to_dataset
from src.data.collator     import build_multimodal_collator, smoke_test
from src.inference.predict import predict_one
from src.engine.decision   import decide

os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
gc.collect()
torch.cuda.empty_cache()

print('✅ imports OK')
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')
print('VRAM total:', f"{torch.cuda.get_device_properties(0).total_memory/1024**3:.1f} GB")

## 1 — Configuración

In [ ]:
DRIVE_ROOT  = '/content/drive/MyDrive/contact-detection'
SPLITS_DIR  = f'{DRIVE_ROOT}/data/splits'
OUTPUTS_DIR = f'{DRIVE_ROOT}/outputs'
CONFIG_PATH = f'{REPO_DIR}/configs/qwen25_3b.yaml'

# Imágenes: Drive (persistente) → local (rápido para training)
DRIVE_IMGS_DIR = f'{DRIVE_ROOT}/data/images'  # se descarga acá una sola vez
LOCAL_IMGS_DIR = '/content/images'             # se copia al inicio de cada sesión

with open(CONFIG_PATH) as f:
    cfg = yaml.safe_load(f)

MODEL_NAME   = cfg['model']['name']
LOAD_4BIT    = cfg['model']['load_in_4bit']
MAX_IMAGES   = cfg['data']['max_images']        # 10
IMG_MAX_SIDE = cfg['data']['img_max_side']       # 512
PROMPT_CHARS = cfg['data']['prompt_max_chars']   # 2500
SEED         = cfg['data']['seed']               # 42

# Ajustá según el tiempo disponible:
#   N_TRAIN=500  → ~25 min descarga + ~15 min training (T4)
#   N_TRAIN=2000 → ~100 min descarga + ~60 min training (T4)
#   N_TRAIN=None → dataset completo (66.9k), solo si imágenes ya están en Drive
N_TRAIN = 1000
N_VAL   = 300

print(f'Modelo  : {MODEL_NAME}')
print(f'LoRA    : r={cfg["peft"]["r"]} / alpha={cfg["peft"]["lora_alpha"]}')
print(f'Épocas  : {cfg["training"]["num_train_epochs"]}')
print(f'N_TRAIN : {N_TRAIN}  |  N_VAL : {N_VAL}')
print(f'Imgs Drive : {DRIVE_IMGS_DIR}')
print(f'Imgs local : {LOCAL_IMGS_DIR}')

## 2 — Dataset

In [ ]:
# Verificar que los splits existen
for split in ['train.csv', 'val.csv']:
    p = Path(SPLITS_DIR) / split
    assert p.exists(), f"❌ No encontrado: {p}. Corré 01_build_splits.ipynb primero."
    df = pd.read_csv(p)
    print(f"  {split}: {len(df):,} filas, {df['RESULT'].mean():.2%} DC")

In [ ]:
# Paso 1: construir dataset con imágenes guardadas en Drive (persistentes)
# Si las imágenes ya existen en Drive, este paso es casi instantáneo
print('Construyendo train dataset (descarga a Drive si faltan imágenes)...')
ds_train = csv_to_dataset(
    csv_path         = f'{SPLITS_DIR}/train.csv',
    img_dir          = f'{DRIVE_IMGS_DIR}/train',
    max_images       = MAX_IMAGES,
    img_max_side     = IMG_MAX_SIDE,
    prompt_max_chars = PROMPT_CHARS,
    seed             = SEED,
    limit            = N_TRAIN,
)

print('\nConstruyendo val dataset...')
ds_val = csv_to_dataset(
    csv_path         = f'{SPLITS_DIR}/val.csv',
    img_dir          = f'{DRIVE_IMGS_DIR}/val',
    max_images       = MAX_IMAGES,
    img_max_side     = IMG_MAX_SIDE,
    prompt_max_chars = PROMPT_CHARS,
    seed             = SEED,
    limit            = N_VAL,
)

print(f'\nDataset en Drive: train={len(ds_train):,}  val={len(ds_val):,}')

In [ ]:
# Paso 2: copiar imágenes de Drive al disco local (SSD rápido)
# Drive  → ~50–100 MB/s (FUSE mount, cuello de botella en training)
# Local  → ~400–500 MB/s (SSD efímero, mucho más rápido para el dataloader)
#
# Estimado: 1k ejemplos × 5 imgs × 100 KB ≈ 500 MB → ~1 seg de copia

import subprocess, shutil
from pathlib import Path

def sync_drive_to_local(drive_dir: str, local_dir: str) -> None:
    """Copia solo los archivos nuevos/modificados (rsync --update)."""
    Path(local_dir).mkdir(parents=True, exist_ok=True)
    result = subprocess.run(
        ['rsync', '-a', '--update', '--info=progress2', f'{drive_dir}/', local_dir],
        capture_output=True, text=True
    )
    if result.returncode != 0:
        print('rsync stderr:', result.stderr[:500])
    else:
        n_files = sum(1 for _ in Path(local_dir).rglob('*.jpg'))
        size_mb = sum(f.stat().st_size for f in Path(local_dir).rglob('*.jpg')) / 1024**2
        print(f'  ✅ {n_files:,} imágenes copiadas a local ({size_mb:.0f} MB)')

print('Sincronizando imágenes Drive → local...')
sync_drive_to_local(f'{DRIVE_IMGS_DIR}/train', f'{LOCAL_IMGS_DIR}/train')
sync_drive_to_local(f'{DRIVE_IMGS_DIR}/val',   f'{LOCAL_IMGS_DIR}/val')

In [ ]:
# Paso 3: remap de paths en el dataset — Drive → local
# El dataset tiene image_path apuntando a Drive; lo remapeamos a local

def remap_to_local(example, drive_prefix: str, local_prefix: str):
    example['image_path'] = [
        p.replace(drive_prefix, local_prefix, 1)
        for p in example['image_path']
    ]
    return example

ds_train = ds_train.map(
    lambda ex: remap_to_local(ex, f'{DRIVE_IMGS_DIR}/train', f'{LOCAL_IMGS_DIR}/train')
)
ds_val = ds_val.map(
    lambda ex: remap_to_local(ex, f'{DRIVE_IMGS_DIR}/val', f'{LOCAL_IMGS_DIR}/val')
)

# Verificar que los paths locales existen
missing = [p for ex in ds_train for p in ex['image_path'] if not Path(p).exists()]
print(f'Paths locales faltantes: {len(missing)}  (debe ser 0)')

from datasets import DatasetDict
ds = DatasetDict({'train': ds_train, 'validation': ds_val})
print(f'✅ Dataset listo con paths locales: train={len(ds["train"]):,}  val={len(ds["validation"]):,}')

In [ ]:
# Smoke check
ex = ds['train'][0]
print('item_id  :', ex['item_id'])
print('Imágenes :', len(ex['image_path']), '→', ex['image_path'][0])
print('Label    :', ex['label'])
print('Answer   :', ex['answer'])

# Distribución de clases
for name, split_ds in ds.items():
    labels = split_ds['label']
    print(f'{name}: {len(labels):,}  DC={sum(labels):,} ({sum(labels)/len(labels):.2%})')

# Histograma de imágenes por listing
n_imgs = [len(ex['image_path']) for ex in ds['train']]
fig, ax = plt.subplots(figsize=(7, 3))
ax.hist(n_imgs, bins=range(1, MAX_IMAGES + 2), edgecolor='white', color='#1565C0')
ax.set_xlabel('Imágenes por listing')
ax.set_title(f'train — media: {np.mean(n_imgs):.1f} imgs/listing')
ax.set_xticks(range(1, MAX_IMAGES + 1))
plt.tight_layout()
plt.show()

## 3 — Modelo + LoRA

In [ ]:
gc.collect()
torch.cuda.empty_cache()

model, tokenizer = FastVisionModel.from_pretrained(
    MODEL_NAME,
    load_in_4bit = LOAD_4BIT,
    device_map   = {'': 0},
    max_memory   = {0: '13GB'},
)

model = FastVisionModel.get_peft_model(
    model,
    r              = cfg['peft']['r'],
    lora_alpha     = cfg['peft']['lora_alpha'],
    lora_dropout   = cfg['peft']['lora_dropout'],
    bias           = cfg['peft']['bias'],
    target_modules = cfg['peft']['target_modules'],
)
model.gradient_checkpointing_enable()

total_p     = sum(p.numel() for p in model.parameters())
trainable_p = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'✅ modelo cargado')
print(f'   Entrenables: {trainable_p:,} / {total_p:,} ({trainable_p/total_p:.3%})')
print(f'   VRAM: {torch.cuda.memory_allocated()/1024**3:.2f} GB alloc / {torch.cuda.memory_reserved()/1024**3:.2f} GB reserved')

## 4 — Collator (con label masking)

In [ ]:
processor = AutoProcessor.from_pretrained(MODEL_NAME)
collator  = build_multimodal_collator(processor)

# Verifica que label_frac << 1 (solo tokens de respuesta tienen loss)
smoke_test(collator, ds['train'], n=2)

## 5 — Entrenamiento

In [ ]:
t_cfg = cfg['training']

steps_per_epoch = len(ds['train']) // (
    t_cfg['per_device_train_batch_size'] * t_cfg['gradient_accumulation_steps']
)
total_steps     = steps_per_epoch * t_cfg['num_train_epochs']
checkpoint_freq = max(10, total_steps // 10)

print(f'steps/epoch    : {steps_per_epoch}')
print(f'total steps    : {total_steps}')
print(f'checkpoint cada: {checkpoint_freq} pasos')

In [ ]:
RUN_NAME   = f"qwen25vl_3b_baseline_{N_TRAIN}ej"
OUTPUT_DIR = f"{OUTPUTS_DIR}/{RUN_NAME}"

training_args = TrainingArguments(
    output_dir                  = OUTPUT_DIR,
    per_device_train_batch_size = t_cfg['per_device_train_batch_size'],
    per_device_eval_batch_size  = t_cfg['per_device_train_batch_size'],
    gradient_accumulation_steps = t_cfg['gradient_accumulation_steps'],
    num_train_epochs            = t_cfg['num_train_epochs'],
    learning_rate               = t_cfg['learning_rate'],
    fp16                        = t_cfg['fp16'],
    logging_steps               = t_cfg.get('logging_steps', 2),
    eval_strategy               = 'steps',
    eval_steps                  = checkpoint_freq,
    save_strategy               = 'steps',
    save_steps                  = checkpoint_freq,
    save_total_limit            = 2,
    load_best_model_at_end      = True,
    metric_for_best_model       = 'eval_loss',
    report_to                   = 'none',
    remove_unused_columns       = False,
)

trainer = Trainer(
    model         = model,
    args          = training_args,
    train_dataset = ds['train'],
    eval_dataset  = ds['validation'],
    data_collator = collator,
)

gc.collect()
torch.cuda.empty_cache()

print('Iniciando entrenamiento...')
trainer.train()
print('✅ Entrenamiento completado')

In [ ]:
# Curva de loss
log_history = trainer.state.log_history
train_logs = [(l['step'], l['loss'])      for l in log_history if 'loss'      in l]
eval_logs  = [(l['step'], l['eval_loss']) for l in log_history if 'eval_loss' in l]

if train_logs:
    fig, ax = plt.subplots(figsize=(9, 4))
    ax.plot(*zip(*train_logs), label='train loss', color='#1565C0')
    if eval_logs:
        ax.plot(*zip(*eval_logs), 'o--', label='eval loss', color='#C62828')
    ax.set_xlabel('Paso')
    ax.set_ylabel('Loss')
    ax.set_title('Curva de loss')
    ax.legend()
    plt.tight_layout()
    plt.show()

## 6 — Guardar en Drive

In [ ]:
import shutil

print(f'Guardando en {OUTPUT_DIR}...')
trainer.model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
processor.save_pretrained(OUTPUT_DIR)      # necesario para reproducir inferencia
shutil.copy(CONFIG_PATH, f'{OUTPUT_DIR}/qwen25_3b.yaml')

print('✅ Guardado')
print('   Archivos:', sorted(f.name for f in Path(OUTPUT_DIR).iterdir() if not f.is_dir()))

## 7 — Evaluación en validación

In [ ]:
N_EVAL = min(N_VAL, len(ds['validation']))
print(f'Corriendo inferencia sobre {N_EVAL} ejemplos de val...')

results = []
for i in range(N_EVAL):
    ex       = ds['validation'][i]
    raw, parsed = predict_one(ex, model=model, processor=processor, max_new_tokens=200)
    decision    = decide(ex['item_id'], parsed)
    results.append({
        'item_id':    ex['item_id'],
        'y_true':     ex['label'],
        'y_pred':     1 if decision.is_dc else 0,
        'raw_label':  decision.raw_label,
        'confidence': decision.confidence,
        'raw_output': raw,
    })

results_df   = pd.DataFrame(results)
parse_errors = (results_df['confidence'] == 'parse_error').sum()
print(f'Parse errors: {parse_errors} / {N_EVAL} ({parse_errors/N_EVAL:.1%})')

In [ ]:
def compute_metrics(y_true, y_pred, beta=2.0):
    y_true, y_pred = np.array(y_true), np.array(y_pred)
    tp = int(((y_true==1)&(y_pred==1)).sum())
    tn = int(((y_true==0)&(y_pred==0)).sum())
    fp = int(((y_true==0)&(y_pred==1)).sum())
    fn = int(((y_true==1)&(y_pred==0)).sum())
    prec = tp/(tp+fp) if (tp+fp)>0 else 0.0
    rec  = tp/(tp+fn) if (tp+fn)>0 else 0.0
    b2   = beta**2
    f2   = (1+b2)*prec*rec/(b2*prec+rec) if (b2*prec+rec)>0 else 0.0
    return dict(tp=tp, tn=tn, fp=fp, fn=fn, precision=prec, recall=rec, f2=f2)

m = compute_metrics(results_df['y_true'], results_df['y_pred'])

print('=' * 42)
print('  MÉTRICAS EN VALIDACIÓN')
print('=' * 42)
print(f'  N eval / parse errors : {N_EVAL} / {parse_errors}')
print(f'  Confusion matrix:')
print(f'              Pred 0   Pred 1')
print(f'  Real 0  :  {m["tn"]:>6}   {m["fp"]:>6}   (TN / FP)')
print(f'  Real 1  :  {m["fn"]:>6}   {m["tp"]:>6}   (FN / TP)')
print()
print(f'  Precision : {m["precision"]:.4f}')
print(f'  Recall    : {m["recall"]:.4f}')
print(f'  F2-score  : {m["f2"]:.4f}  ← métrica principal')
print('=' * 42)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))

# Confusion matrix
ax = axes[0]
cm = np.array([[m['tn'], m['fp']], [m['fn'], m['tp']]])
im = ax.imshow(cm, cmap='Blues')
ax.set_xticks([0,1]); ax.set_yticks([0,1])
ax.set_xticklabels(['Pred 0','Pred 1'])
ax.set_yticklabels(['Real 0','Real 1'])
for (i,j), v in np.ndenumerate(cm):
    ax.text(j, i, str(v), ha='center', va='center', fontsize=14,
            color='white' if v > cm.max()/2 else 'black')
ax.set_title('Confusion Matrix')
plt.colorbar(im, ax=ax)

# Barplot de métricas
ax2 = axes[1]
vals = [m['precision'], m['recall'], m['f2']]
bars = ax2.bar(['Precision','Recall','F2-score'], vals,
               color=['#42A5F5','#66BB6A','#FFA726'])
ax2.set_ylim(0, 1.1)
ax2.set_title('Métricas')
for bar, v in zip(bars, vals):
    ax2.text(bar.get_x()+bar.get_width()/2, v+0.02, f'{v:.3f}',
             ha='center', va='bottom', fontsize=11)
ax2.axhline(y=0.8, color='red', linestyle='--', alpha=0.5, label='target F2=0.8')
ax2.legend()

plt.tight_layout()
plt.show()

## 8 — Análisis de errores

In [ ]:
fp_df = results_df[(results_df['y_true']==0) & (results_df['y_pred']==1)]
fn_df = results_df[(results_df['y_true']==1) & (results_df['y_pred']==0)]

print(f'FP: {len(fp_df)}  (modelo dice DC, no es DC)')
print(f'FN: {len(fn_df)}  (modelo dice no-DC, sí es DC)  ← más crítico para F2')

print('\n--- Falsos Negativos (primeros 5) ---')
for _, row in fn_df.head(5).iterrows():
    print(f"  {row['item_id']}  conf={row['confidence']}  raw={row['raw_output'][:180]}")
    print()

In [ ]:
# Guardar resultados
results_df.to_csv(f'{OUTPUT_DIR}/eval_results.csv', index=False)
with open(f'{OUTPUT_DIR}/metrics.json', 'w') as f:
    json.dump({
        'n_train': N_TRAIN, 'n_eval': N_EVAL,
        'parse_errors': int(parse_errors),
        **{k: round(float(v), 4) for k, v in m.items()},
    }, f, indent=2)

print(f'✅ eval_results.csv y metrics.json guardados en {OUTPUT_DIR}')

---
**Siguiente paso:** analizar los FN, ajustar prompt o aumentar `N_TRAIN` para el siguiente sprint.